## Segmentación, embeddings y entrenamiento MLP-MIL

**Guía de ejecución:** Ejecutar en orden; algunas operaciones generan embeddings y checkpoints grandes. Recuperar primero los artefactos con scripts/import_colab_archive.py. El notebook usa paquetes compartidos de src/bias_detector.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


**Versión adaptada para el repositorio unificado.** Los módulos reutilizables se importan desde `src/bias_detector/`; los datos pesados se guardan bajo `data/` y no se descargan automáticamente. Ejecutar el notebook en orden.

In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


### Configuración

In [ ]:
import re
import ast
import json
import bisect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from pathlib import Path

In [ ]:
# Datos del corpus (provenían de /entrenamiento/ de Google Drive)
BASE_DIR = DATA_DIR / 'modeling'
CORPUS_DATA = CANONICAL_CORPUS_DATA
ANNOTATION_DATA = FINAL_ANNOTATIONS_DATA
require_file(CORPUS_DATA, 'corpus de entrenamiento')
require_file(ANNOTATION_DATA, 'etiquetas finales')

#### Check versión de pandas

In [ ]:
# Dependencias del proyecto: instalar una vez desde la raíz con `%pip install -e ".[notebooks]"`.


#### Check uso de GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Crear dos matrices directamente en la GPU
a = torch.randn(4000, 4000, device=device)
b = torch.randn(4000, 4000, device=device)

# Ejecutar una multiplicación en la GPU
c = torch.matmul(a, b)

print("Dispositivo utilizado:", c.device)

#### Carga de datos

In [ ]:
df_corpus = pd.read_pickle(CORPUS_DATA)
df_annotations = pd.read_pickle(ANNOTATION_DATA)

### Análisis de viabilidad de contexto y tokens

In [ ]:
# Configuración
LIMITS = [512, 1024, 2048, 4096]

tokenizer = AutoTokenizer.from_pretrained(
    "allenai/longformer-base-4096",
    use_fast=True
)

In [ ]:
# SELECCIONAR ANOTACIONES POSITIVAS
positive = df_annotations.loc[
    df_annotations["final_label"] == "PRESENT",
    [
        "conversation_id",
        "bias",
        "final_trigger_turns",
        "final_evidence_turns"
    ]
].copy()

positive = positive.merge(
    df_corpus[["conversation_id", "conversation_text"]],
    on="conversation_id",
    how="inner",
    validate="many_to_one"
)

In [ ]:
# LOCALIZAR TURNOS EN TOKENS
TURN_PATTERN = re.compile(
    r"\[TURN\s+(\d+)\s*\|\s*(USER|LLM)\]"
)

def get_turn_positions(text):

    matches = list(TURN_PATTERN.finditer(text))

    encoding = tokenizer(
        text,
        truncation=False,
        return_offsets_mapping=True,
        verbose=False
    )

    offsets = encoding["offset_mapping"]
    positions = {}

    for i, match in enumerate(matches):

        turn_id = int(match.group(1))
        start = match.end()

        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(text)
        )

        # Índices de tokens correspondientes al contenido
        indices = [
            j for j, (a, b) in enumerate(offsets)
            if b > start and a < end and b > a
        ]

        if indices:
            positions[turn_id] = max(indices)

    return len(encoding["input_ids"]), positions


# Calculamos una sola vez por conversación
conversation_positions = {}

for row in positive[
    ["conversation_id", "conversation_text"]
].drop_duplicates("conversation_id").itertuples(index=False):

    conversation_positions[row.conversation_id] = (
        get_turn_positions(row.conversation_text)
    )

In [ ]:
# CALCULAR PÉRDIDA DE TURNOS
results = []

for row in positive.itertuples(index=False):

    n_tokens, positions = conversation_positions[
        row.conversation_id
    ]

    for limit in LIMITS:

        result = {
            "conversation_id": row.conversation_id,
            "bias": row.bias,
            "limit": limit,
            "n_tokens": n_tokens
        }

        for field, turns in [
            ("trigger", row.final_trigger_turns),
            ("evidence", row.final_evidence_turns)
        ]:

            # Excluir listas vacías o identificadores ausentes
            valid = bool(turns) and all(
                t in positions for t in turns
            )

            lost = (
                sum(positions[t] >= limit for t in turns)
                if valid else None
            )

            result[f"{field}_valid"] = valid
            result[f"{field}_any_lost"] = (
                lost > 0 if valid else None
            )
            result[f"{field}_all_lost"] = (
                lost == len(turns) if valid else None
            )

        results.append(result)

df_coverage = pd.DataFrame(results)

In [ ]:
# RESULTADOS GLOBALES
for field in ["trigger", "evidence"]:

    data = df_coverage[
        df_coverage[f"{field}_valid"]
    ]

    summary = data.groupby("limit").agg(
        n_anotaciones=(f"{field}_any_lost", "count"),
        pct_alguna_perdida=(f"{field}_any_lost", "mean"),
        pct_todas_perdidas=(f"{field}_all_lost", "mean")
    )

    summary[
        ["pct_alguna_perdida", "pct_todas_perdidas"]
    ] *= 100

    print(f"\n=== {field.upper()} ===")
    display(summary.round(2))

In [ ]:
# RESULTADOS POR SESGO
by_bias = (
    df_coverage[df_coverage["trigger_valid"]]
    .groupby(["bias", "limit"])
    .agg(
        n_anotaciones=("trigger_any_lost", "count"),
        pct_alguna_perdida=("trigger_any_lost", "mean"),
        pct_todas_perdidas=("trigger_all_lost", "mean")
    )
    .reset_index()
)

by_bias[
    ["pct_alguna_perdida", "pct_todas_perdidas"]
] *= 100

print("\n=== PÉRDIDA DE TRIGGERS POR SESGO ===")
display(by_bias.round(2))

In [ ]:
# VISUALIZACIÓN
trigger_summary = (
    df_coverage[df_coverage["trigger_valid"]]
    .groupby("limit")[
        ["trigger_any_lost", "trigger_all_lost"]
    ]
    .mean() * 100
)

trigger_summary.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Pérdida de turnos desencadenantes")
plt.xlabel("Límite de tokens")
plt.ylabel("Anotaciones afectadas (%)")
plt.ylim(0, 100)
plt.legend([
    "Al menos un turno perdido",
    "Todos los turnos perdidos"
])
plt.tight_layout()
plt.show()

### Segmentación de ventanas

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/segmentation.py
from bias_detector.segmentation import (
    TURN_RE, SegmentationConfig, create_tokenizer, normalize_conversation, segment_conversation, segment_corpus
)


In [ ]:
# Configuración inicial
config = SegmentationConfig(
    model_name="allenai/longformer-base-4096",
    max_length=2048,
    overlap=128,
    min_boundary_fraction=0.75
)

# Tokenizador
tokenizer = create_tokenizer(config)

In [ ]:
# Prueba con 10 conversaciones
df_test = df_corpus.head(10).copy()

df_windows_test = segment_corpus(
    df_test,
    tokenizer,
    config
)

In [ ]:
# Inspeccionar resultados
display(
    df_windows_test[
        [
            "conversation_id",
            "window_id",
            "n_tokens",
            "token_start",
            "token_end",
            "turn_ids"
        ]
    ]
)

In [ ]:
# ---- Comprobaciones ----
# Longitud de las ventanas
assert df_windows_test["n_tokens"].max() <= 2048

# Todas las conversaciones deben tener al menos una ventana
assert (
    df_windows_test["conversation_id"].nunique()
    == len(df_test)
)

# Identificadores únicos de ventanas por conversación
assert not df_windows_test.duplicated(
    ["conversation_id", "window_id"]
).any()

print("Validaciones superadas.")

print(
    "Total de ventanas:",
    len(df_windows_test)
)

print(
    "Media de ventanas por conversación:",
    round(len(df_windows_test) / len(df_test), 2)
)


In [ ]:
# Segmentación para el corpus completo
df_windows = segment_corpus(
    df_corpus,
    tokenizer,
    config
)

# Número de ventanas por conversación
windows_per_conversation = (
    df_windows.groupby("conversation_id")
    .size()
    .rename("n_windows")
)

print("=== ESTADÍSTICAS DE SEGMENTACIÓN ===")

display(
    windows_per_conversation.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2)
)

print("\nTotal de ventanas:", len(df_windows))

print(
    "Conversaciones con múltiples ventanas:",
    (windows_per_conversation > 1).sum()
)

print(
    "Porcentaje con múltiples ventanas:",
    round(
        (windows_per_conversation > 1).mean() * 100,
        2
    ),
    "%"
)

print(
    "Máximo de tokens por ventana:",
    df_windows["n_tokens"].max()
)


In [ ]:
# Número de ventanas por conversación
windows_per_conversation = (
    df_windows.groupby("conversation_id").size()
)

# Todas las conversaciones deben aparecer
assert (
    windows_per_conversation.size
    == df_corpus["conversation_id"].nunique()
)

# No debe haber ventanas vacías
assert (df_windows["n_tokens"] > 2).all()

# Todas las ventanas deben respetar el límite
assert (df_windows["n_tokens"] <= 2048).all()

# Identificadores únicos dentro de cada conversación
assert not df_windows.duplicated(
    ["conversation_id", "window_id"]
).any()

print("Validaciones superadas.")

### Extracción de embeddings

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/embedding_extractor.py
from bias_detector.embedding_extractor import (
    KEYS, COLUMNS, save_windows, load_windows, load_encoder, extract_embeddings, load_embeddings
)


#### Guardado de las segmentaciones

In [ ]:
# Dependencias: instalar una sola vez desde la raíz del repo con `%pip install -e ".[notebooks]"`.

In [ ]:
# No es necesario montar Google Drive: PROJECT_ROOT y rutas centralizadas están configurados arriba.

In [ ]:
# Segmentación, embeddings y splits dentro de data/model_data/
BASE_DIR = MODEL_DATA_DIR
SEGMENTATION_DIR = BASE_DIR / 'segmentation'
EMBEDDINGS_DIR = BASE_DIR / 'embeddings'
BASE_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
# Guardar la segmentación que ya hemos generado (no se sobreescribe una segmentación ya guardada)
save_windows(
    df_windows=df_windows,
    folder=SEGMENTATION_DIR,
    segmentation_config=config
)

print("Segmentación guardada correctamente.")

In [ ]:
# Para comprobar que podemos recuperarla
df_windows_saved, windows_manifest = load_windows(
    SEGMENTATION_DIR
)

print("Ventanas recuperadas:", len(df_windows_saved))
print("Conversaciones:", df_windows_saved["conversation_id"].nunique())

display(df_windows_saved.head(3))

#### Prueba de extracción con la T4

In [ ]:
# Comprobamos que la GPU está disponible
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM total (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        )
    )

In [ ]:
# Pequeña prueba de extracción
# 10 ventanas para validar el extractor
df_sample = df_windows_saved.head(10).copy()

# Manifiesto temporal para esta prueba
test_manifest = {
    **windows_manifest,
    "num_windows": len(df_sample),
    "sha256": "test_first_10_windows"
}

TEST_DIR = BASE_DIR / "embeddings_test"

extract_embeddings(
    df_windows=df_sample,
    windows_manifest=test_manifest,
    output_dir=TEST_DIR,
    model_name="allenai/longformer-base-4096",
    batch_size=1,
    shard_size=10,
    fp16=True
)

# Recuperar los embeddings
embeddings_test, window_keys = load_embeddings(TEST_DIR)

print("Forma de la matriz:", embeddings_test.shape)
print("Número de identificadores:", len(window_keys))


#### Extracción del corpus completo

In [ ]:
# Extracción del corpus completo
extract_embeddings(
    df_windows=df_windows_saved,
    windows_manifest=windows_manifest,
    output_dir=EMBEDDINGS_DIR,
    model_name="allenai/longformer-base-4096",
    batch_size=1,
    shard_size=128,
    fp16=True
)

#### Reconstruir y validar los embeddings

In [ ]:
embeddings, window_keys = load_embeddings(EMBEDDINGS_DIR)

# Comprobaciones
assert embeddings.shape == (len(df_windows_saved), 768)
assert np.isfinite(embeddings).all()
assert len(window_keys) == len(set(window_keys))

# Verificar correspondencia con las ventanas originales
expected_keys = (
    df_windows_saved["conversation_id"].astype(str)
    + "::"
    + df_windows_saved["window_id"].astype(str)
).to_numpy()

assert np.array_equal(window_keys, expected_keys)

print("Embeddings recuperados y validados.")
print("Dimensiones:", embeddings.shape)
print("Tipo de dato:", embeddings.dtype)
print("Memoria (MiB):", round(embeddings.nbytes / 1024**2, 2))

### Arquitectura del clasificador

A partir de este punto comenzaría el segundo notebook.

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/mil_model.py
from bias_detector.mil_model import (
    MILConfig, MaxMILClassifier, build_label_matrix, align_embeddings_and_labels
)


In [ ]:
# Configuración
config_mil = MILConfig()

# Matriz de etiquetas
df_labels_mil = build_label_matrix(
    df_annotations,
    biases=config_mil.biases
)

# Alinear embeddings y etiquetas
X, bag_ids, y, conversation_ids = (
    align_embeddings_and_labels(
        embeddings,
        window_keys,
        df_labels_mil
    )
)

In [ ]:
# Crear clasificador
model = MaxMILClassifier()

In [ ]:
# Primera prueba de predicción
X_tensor = torch.from_numpy(X)
bag_tensor = torch.from_numpy(bag_ids)

with torch.no_grad():
    logits = model(X_tensor, bag_tensor)
    probabilities = torch.sigmoid(logits)

print("Embeddings:", X.shape)
print("Etiquetas:", y.shape)
print("Logits:", logits.shape)
print("Probabilidades:", probabilities.shape)
print("Parámetros entrenables:", sum(
    p.numel() for p in model.parameters()
))

### Splits conjunto de entrenamiento, valdiación y prueba

#### Configuración

In [ ]:
%pip install -q iterative-stratification

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/mil_splits.py
from bias_detector.mil_splits import (
    SplitConfig, create_splits, validate_splits, split_report, save_splits, load_splits, select_bags
)


#### Partición del corpus

In [ ]:
# Generamos matriz multietiqueta
config_mil = MILConfig()

df_labels_mil = build_label_matrix(
    df_annotations,
    biases=config_mil.biases
)

# Configuración
split_config = SplitConfig(
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
)

In [ ]:
# Generamos las particiones
df_splits = create_splits(
    labels=df_labels_mil,
    config=split_config
)

In [ ]:
# Comprobamos distribución
report = split_report(df_splits, df_labels_mil)

display(report)

#### Guardado de las particiones

In [ ]:
SPLITS_DIR = BASE_DIR / "splits"

In [ ]:
# Guardamos las particiones
save_splits(
    splits=df_splits,
    labels=df_labels_mil,
    folder=SPLITS_DIR,
    config=split_config
)

print("Particiones guardadas correctamente.")

In [ ]:
# En sesiones posteriores podemos recuperarlas
df_splits = load_splits(
    SPLITS_DIR,
    labels=df_labels_mil
)

In [ ]:
masks = {
    split: select_bags(
        bag_ids,
        conversation_ids,
        df_splits,
        split
    )
    for split in ["train", "val", "test"]
}

for split, mask in masks.items():
    print(
        f"{split}: "
        f"{len(np.unique(bag_ids[mask]))} conversaciones, "
        f"{mask.sum()} ventanas"
    )

# Toda ventana debe pertenecer a exactamente una partición
assert np.all(
    sum(mask.astype(int) for mask in masks.values()) == 1
)

print("Todas las ventanas están correctamente asignadas.")


### Entrenamiendo del clasificador MLP-MIL

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/mil_training.py
from bias_detector.mil_training import (
    TrainingConfig, set_seed, partition_arrays, BagDataset, collate_bags, compute_pos_weight, train_mil, load_best_model, predict_bags
)


In [ ]:
# Entrenamiento
train_groups, y_train, train_ids = partition_arrays(
    X, bag_ids, y, conversation_ids,
    df_splits, split="train"
)

print("=== ENTRENAMIENTO ===")
print("Conversaciones:", len(train_groups))
print("Ventanas:", sum(len(g) for g in train_groups))
print("Etiquetas:", y_train.shape)

In [ ]:
# Validación
val_groups, y_val, val_ids = partition_arrays(
    X, bag_ids, y, conversation_ids,
    df_splits, split="val"
)

print("\n=== VALIDACIÓN ===")
print("Conversaciones:", len(val_groups))
print("Ventanas:", sum(len(g) for g in val_groups))
print("Etiquetas:", y_val.shape)

Antes de entrenar, vamos a calcular los pesos que utilizará la función de pérdida.

Recordemos que nuestro corpus está muy desbalanceado. Para evitar utilizar las razones completas entre negativos y positivos, aplicaremos la raíz cuadrada de la división entre el número de ejemplos positivos y negativos de cada sesgo.

In [ ]:
weights = compute_pos_weight(
    y_train,
    strategy="sqrt"
)

for bias, weight in zip(config_mil.biases, weights):
    print(f"{bias}: {weight.item():.4f}")

#### Entrenamiento del clasificador

In [ ]:
# Directorio del experimento
TRAINING_DIR = BASE_DIR / "training" / "mlp_mil_v1"

In [ ]:
# Configuración
training_config = TrainingConfig(
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=0.01,
    patience=12,
    min_delta=1e-4,
    seed=42,
    weight_strategy="sqrt",
    grad_clip=1.0,
    device="cpu"
)

In [ ]:
# Inicialización reproducible
set_seed(training_config.seed)

# Usamos la clase MLP que ya validaste
model = MaxMILClassifier(
    input_dim=768,
    hidden_dim=256,
    bottleneck_dim=64,
    num_labels=4,
    dropout=0.3
)

print(
    "Parámetros entrenables:",
    sum(p.numel() for p in model.parameters())
)

In [ ]:
# Entrenamiento
history = train_mil(
    model=model,
    train_groups=train_groups,
    y_train=y_train,
    val_groups=val_groups,
    y_val=y_val,
    output_dir=TRAINING_DIR,
    config=training_config,
    biases=tuple(config_mil.biases)
)

#### Análisis de las curvas de aprendizaje

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    history["epoch"],
    history["train_loss"],
    label="Entrenamiento"
)

plt.plot(
    history["epoch"],
    history["val_loss"],
    label="Validación"
)

plt.xlabel("Época")
plt.ylabel("BCE ponderada")
plt.title("Evolución del entrenamiento MLP-MIL")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_epoch = history.loc[
    history["val_loss"].idxmin()
]

print("Mejor época:", int(best_epoch["epoch"]))
print("Mejor pérdida de validación:", best_epoch["val_loss"])

#### Recuperación del mejor modelo y generación de predicciones de validación

In [ ]:
# Recuperar mejor modelo
model, checkpoint_info = load_best_model(
    model,
    TRAINING_DIR,
    device="cpu"
)

print("Mejor época:", checkpoint_info["epoch"])
print("Pérdida:", checkpoint_info["val_loss"])

# Predicciones de validación
val_probabilities = predict_bags(
    model,
    val_groups,
    batch_size=32,
    device="cpu"
)

print("Dimensiones:", val_probabilities.shape)
print("Mínimo:", val_probabilities.min())
print("Máximo:", val_probabilities.max())

In [ ]:
# Veamos algunos ejemplos de primeras predicciones
df_val_predictions = pd.DataFrame(
    val_probabilities,
    columns=config_mil.biases
)

df_val_predictions.insert(
    0, "conversation_id", val_ids
)

display(df_val_predictions.head(10))

In [ ]:
# Analizamos la distribución de puntuaciones
def plot_score_distributions(
    probabilities,
    labels,
    bias_names
):
    """
    Representa la distribución de puntuaciones
    para ejemplos positivos y negativos de cada sesgo.
    """

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for j, bias in enumerate(bias_names):

        ax = axes[j]

        scores_positive = probabilities[labels[:, j] == 1, j]
        scores_negative = probabilities[labels[:, j] == 0, j]

        bins = np.linspace(0, 1, 31)

        ax.hist(
            scores_negative,
            bins=bins,
            alpha=0.6,
            density=True,
            label=f"ABSENT (n={len(scores_negative)})"
        )

        ax.hist(
            scores_positive,
            bins=bins,
            alpha=0.6,
            density=True,
            label=f"PRESENT (n={len(scores_positive)})"
        )

        ax.axvline(
            0.5,
            linestyle="--",
            label="Umbral 0.5"
        )

        ax.set_title(bias)
        ax.set_xlabel("Puntuación del modelo")
        ax.set_ylabel("Densidad")
        ax.legend()

    plt.tight_layout()
    plt.show()


plot_score_distributions(
    val_probabilities,
    y_val,
    config_mil.biases
)

In [ ]:
# Obtenemos estadísticas numéricas
def summarize_score_distributions(
    probabilities,
    labels,
    bias_names
):
    rows = []

    for j, bias in enumerate(bias_names):

        for label_value, label_name in [
            (0, "ABSENT"),
            (1, "PRESENT")
        ]:

            scores = probabilities[
                labels[:, j] == label_value, j
            ]

            rows.append({
                "bias": bias,
                "label": label_name,
                "n": len(scores),
                "mean": scores.mean(),
                "median": np.median(scores),
                "p25": np.percentile(scores, 25),
                "p75": np.percentile(scores, 75),
                "p90": np.percentile(scores, 90),
                "min": scores.min(),
                "max": scores.max()
            })

    return pd.DataFrame(rows)


df_score_summary = summarize_score_distributions(
    val_probabilities,
    y_val,
    config_mil.biases
)

display(df_score_summary.round(4))

In [ ]:
# Capacidad de discriminación antes de optimizar umbrales
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

def evaluate_discrimination(
    probabilities,
    labels,
    bias_names
):
    results = []

    for j, bias in enumerate(bias_names):

        y_true = labels[:, j]
        y_score = probabilities[:, j]

        results.append({
            "bias": bias,
            "n_positive": int(y_true.sum()),
            "prevalence": float(y_true.mean()),
            "roc_auc": roc_auc_score(y_true, y_score),
            "average_precision": average_precision_score(
                y_true, y_score
            )
        })

    return pd.DataFrame(results)


df_discrimination = evaluate_discrimination(
    val_probabilities,
    y_val,
    config_mil.biases
)

display(df_discrimination.round(4))

### Optimización de los umbrales de clasificación

Búsqueda sobre las puntuaciones reales del modelo, en lugar de probar únicamente valores como 0,1, 0,2 o 0,3. Esto permite evaluar todos los puntos en los que cambia la clasificación de una conversación.

Si varios umbrales consiguen el mismo F1, seleccionaremos el que produzca mayor precisión y, si persiste el empate, el umbral más alto.

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/threshold_optimization.py
from bias_detector.threshold_optimization import (
    ThresholdConfig, metrics_at_thresholds, optimize_thresholds, apply_thresholds, save_thresholds, load_thresholds, plot_precision_recall, plot_confusion_matrices
)


#### Optimizar umbrales

In [ ]:
# Optimización independiente para cada sesgo
thresholds, df_threshold_metrics, overall_metrics = (
    optimize_thresholds(
        y_val=y_val,
        val_probabilities=val_probabilities,
        bias_names=config_mil.biases
    )
)

print("=== UMBRALES ÓPTIMOS ===")

for bias, threshold in zip(config_mil.biases, thresholds):
    print(f"{bias}: {threshold:.6f}")

print("\n=== MÉTRICAS DE VALIDACIÓN ===")
display(df_threshold_metrics.round(4))

print("\n=== MÉTRICAS GLOBALES ===")

for metric, value in overall_metrics.items():
    print(f"{metric}: {value:.4f}")

#### Comparación con umbral convencional de 0,5

In [ ]:
# Evaluación con umbral estándar
baseline_thresholds = np.full(4, 0.5)

df_baseline, baseline_overall = metrics_at_thresholds(
    y_true=y_val,
    scores=val_probabilities,
    bias_names=config_mil.biases,
    thresholds=baseline_thresholds
)

# Comparación de métricas
comparison = pd.DataFrame({
    "bias": config_mil.biases,
    "threshold_optimized": thresholds,
    "f1_default": df_baseline["f1"].to_numpy(),
    "f1_optimized": df_threshold_metrics["f1"].to_numpy(),
    "precision_optimized": df_threshold_metrics["precision"].to_numpy(),
    "recall_optimized": df_threshold_metrics["recall"].to_numpy()
})

comparison["f1_improvement"] = (
    comparison["f1_optimized"] -
    comparison["f1_default"]
)

display(comparison.round(4))

print(
    "Macro-F1 con umbral 0.5:",
    round(baseline_overall["macro_f1"], 4)
)

print(
    "Macro-F1 optimizado:",
    round(overall_metrics["macro_f1"], 4)
)

In [ ]:
# Visualizar curvas precision-recall
fig = plot_precision_recall(
    y_true=y_val,
    scores=val_probabilities,
    bias_names=config_mil.biases,
    thresholds=thresholds
)

plt.show()

In [ ]:
# Matrices de confusión
fig = plot_confusion_matrices(
    y_true=y_val,
    scores=val_probabilities,
    bias_names=config_mil.biases,
    thresholds=thresholds
)

plt.show()

#### Guardado de umbrales


In [ ]:
THRESHOLDS_DIR = TRAINING_DIR / "thresholds"

In [ ]:
save_thresholds(
    folder=THRESHOLDS_DIR,
    thresholds=thresholds,
    bias_names=config_mil.biases,
    report=df_threshold_metrics,
    overall=overall_metrics,
    metadata={
        "model_experiment": "mlp_mil_v1",
        "split": "val",
        "n_conversations": len(y_val)
    }
)

print("Umbrales guardados correctamente.")

In [ ]:
# Podremos recuperar los umbrales como sigue
thresholds, metadata = load_thresholds(
    THRESHOLDS_DIR,
    bias_names=config_mil.biases
)

### Comparación del modelo con otras configuraciones de ponderaciones

El código siguiente se podría integrar con el código anterior del clasificador.

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/mil_training.py
from bias_detector.mil_training import (
    TrainingConfig, set_seed, partition_arrays, BagDataset, collate_bags, compute_pos_weight, train_mil, load_best_model, predict_bags
)


In [ ]:
# Comprobamos los pesos según las diferentes estrategias
for strategy in ["none", "quarter", "sqrt"]:
    weights = compute_pos_weight(y_train, strategy=strategy)

    print(f"\n{strategy}:")
    for bias, weight in zip(config_mil.biases, weights):
        print(f"  {bias}: {weight.item():.4f}")

In [ ]:
# Entrenamos las dos nuevas versiones
EXPERIMENTS = {
    "mlp_mil_v2": "none",
    "mlp_mil_v3": "quarter"
}

experiment_results = {}
bias_names = tuple(config_mil.biases)

for experiment_name, weight_strategy in EXPERIMENTS.items():

    print(f"\n{'=' * 50}")
    print(f"EXPERIMENTO: {experiment_name}")
    print(f"Ponderación: {weight_strategy}")
    print(f"{'=' * 50}")

    experiment_dir = (
        BASE_DIR / "training" / experiment_name
    )

    # Misma inicialización para ambos experimentos
    set_seed(42)

    model = MaxMILClassifier(
        input_dim=768,
        hidden_dim=256,
        bottleneck_dim=64,
        num_labels=4,
        dropout=0.3
    )

    training_config = TrainingConfig(
        epochs=100,
        batch_size=32,
        learning_rate=1e-3,
        weight_decay=0.01,
        patience=12,
        min_delta=1e-4,
        seed=42,
        weight_strategy=weight_strategy,
        grad_clip=1.0,
        device="cpu"
    )

    # Entrenamiento y early stopping
    history = train_mil(
        model=model,
        train_groups=train_groups,
        y_train=y_train,
        val_groups=val_groups,
        y_val=y_val,
        output_dir=experiment_dir,
        config=training_config,
        biases=bias_names
    )

    # Recuperar los mejores pesos
    model, checkpoint = load_best_model(
        model,
        experiment_dir,
        device="cpu"
    )

    # Predicciones sobre validación
    probabilities = predict_bags(
        model,
        val_groups,
        batch_size=32,
        device="cpu"
    )

    # Optimización independiente de umbrales
    thresholds, metrics, overall = optimize_thresholds(
        y_val=y_val,
        val_probabilities=probabilities,
        bias_names=bias_names
    )

    # Guardar los umbrales del experimento
    save_thresholds(
        folder=experiment_dir / "thresholds",
        thresholds=thresholds,
        bias_names=bias_names,
        report=metrics,
        overall=overall,
        metadata={
            "model_experiment": experiment_name,
            "weight_strategy": weight_strategy,
            "split": "val",
            "n_conversations": len(y_val)
        }
    )

    experiment_results[experiment_name] = {
        "history": history,
        "best_epoch": checkpoint["epoch"],
        "metrics": metrics,
        "overall": overall,
        "thresholds": thresholds,
        "probabilities": probabilities
    }

    print(f"\nMejor época: {checkpoint['epoch']}")
    print(f"Macro-F1: {overall['macro_f1']:.4f}")
    print(f"Micro-F1: {overall['micro_f1']:.4f}")

print("\nExperimentos finalizados.")


In [ ]:
# Comparaciones
# Incorporar resultados existentes de v1
experiment_results["mlp_mil_v1"] = {
    "metrics": df_threshold_metrics,
    "overall": overall_metrics,
    "thresholds": thresholds  # Umbrales originales de v1
}

comparison_rows = []

for name, result in experiment_results.items():

    metrics = result["metrics"]
    overall = result["overall"]

    comparison_rows.append({
        "experiment": name,
        "macro_f1": overall["macro_f1"],
        "micro_f1": overall["micro_f1"],
        "macro_precision": overall["macro_precision"],
        "macro_recall": overall["macro_recall"],
        "mean_AP": metrics["average_precision"].mean(),
        "total_TP": int(metrics["tp"].sum()),
        "total_FP": int(metrics["fp"].sum()),
        "total_FN": int(metrics["fn"].sum())
    })

df_comparison = pd.DataFrame(comparison_rows)

display(
    df_comparison.sort_values(
        "macro_f1",
        ascending=False
    ).round(4)
)

# Comparación detallada de F1 por sesgo
df_f1_comparison = pd.DataFrame({
    name: result["metrics"].set_index("bias")["f1"]
    for name, result in experiment_results.items()
})

print("\n=== F1 POR SESGO ===")
display(df_f1_comparison.round(4))

In [ ]:
# Persistir comparación
comparison_path = BASE_DIR / "training" / "weight_comparison.csv"

df_comparison.to_csv(
    comparison_path,
    index=False
)

### Evaluación del conjunto de prueba

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/final_evaluation.py
from bias_detector.final_evaluation import (
    evaluate_multilabel, predictions_dataframe, save_final_evaluation, plot_confusion_matrices
)


Recuperamos modelo y umbrales de v3

In [ ]:
# Experimento seleccionado
EXPERIMENT_DIR = BASE_DIR / "training" / "mlp_mil_v3"
EVALUATION_DIR = EXPERIMENT_DIR / "final_evaluation"

bias_names = tuple(config_mil.biases)

# Recuperar umbrales ya optimizados en validación
thresholds_v3, threshold_metadata = load_thresholds(
    EXPERIMENT_DIR / "thresholds",
    bias_names=bias_names
)

assert threshold_metadata["selection_dataset"] == "validation"

print("Umbrales definitivos:")
for bias, threshold in zip(bias_names, thresholds_v3):
    print(f"{bias}: {threshold:.6f}")

Recuperamos la misma arquitectura MLP entrenada anteriormente.

In [ ]:
# Utilizar la clase MaxMILClassifier MLP ya definida
model_v3 = MaxMILClassifier(
    input_dim=768,
    hidden_dim=256,
    bottleneck_dim=64,
    num_labels=4,
    dropout=0.3
)

model_v3, checkpoint_v3 = load_best_model(
    model=model_v3,
    output_dir=EXPERIMENT_DIR,
    device="cpu"
)

# Comprobar el experimento cargado
assert checkpoint_v3["training_config"]["weight_strategy"] == "quarter"
assert tuple(checkpoint_v3["biases"]) == bias_names

print("Modelo v3 recuperado correctamente.")
print("Mejor época:", checkpoint_v3["epoch"])
print("Pérdida de validación:", checkpoint_v3["val_loss"])


Preparación del conjunto de prueba

In [ ]:
test_groups, y_test, test_ids = partition_arrays(
    X=X,
    bag_ids=bag_ids,
    y=y,
    conversation_ids=conversation_ids,
    splits=df_splits,
    split="test"
)

print("Conversaciones de prueba:", len(test_groups))
print("Ventanas de prueba:", sum(len(g) for g in test_groups))
print("Etiquetas:", y_test.shape)

assert len(test_groups) == 1452
assert sum(len(g) for g in test_groups) == 1727
assert y_test.shape == (1452, 4)

Obtención de predicciones para el conjunto de prueba.

In [ ]:
test_probabilities = predict_bags(
    model=model_v3,
    groups=test_groups,
    batch_size=32,
    device="cpu"
)

print("Forma de las predicciones:", test_probabilities.shape)

assert test_probabilities.shape == y_test.shape
assert np.isfinite(test_probabilities).all()

A partir de este punto, todas las métricas se calcularán usando estos resultados, sin modificar las puntuaciones ni los umbrales.

#### Evaluación multietiqueta completa

In [ ]:
df_test_metrics, test_overall, test_matrices, test_predictions = (
    evaluate_multilabel(
        y_true=y_test,
        scores=test_probabilities,
        thresholds=thresholds_v3,
        bias_names=bias_names,
        beta=2.0
    )
)

print("=== MÉTRICAS POR SESGO ===")
display(df_test_metrics.round(4))

print("\n=== MÉTRICAS GLOBALES ===")
display(
    pd.DataFrame(
        test_overall.items(),
        columns=["metric", "value"]
    ).round(4)
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def plot_test_confusion_matrices(df_metrics):
    """Representa las matrices de confusión por sesgo."""

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))

    for ax, (_, row) in zip(axes.flat, df_metrics.iterrows()):

        cm = np.array([
            [row["tn"], row["fp"]],
            [row["fn"], row["tp"]]
        ], dtype=int)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["ABSENT", "PRESENT"]
        )

        disp.plot(
            ax=ax,
            cmap="Blues",
            colorbar=False,
            values_format="d"
        )

        ax.set_title(row["bias"])

    plt.tight_layout()
    return fig


fig = plot_test_confusion_matrices(df_test_metrics)
plt.show()


#### Guardado de métricas

In [ ]:
df_test_predictions = predictions_dataframe(
    conversation_ids=test_ids,
    y_true=y_test,
    scores=test_probabilities,
    predictions=test_predictions,
    bias_names=bias_names
)

display(df_test_predictions.head())

# Guardar los resultados finales
saved_files = save_final_evaluation(
    folder=EVALUATION_DIR,
    per_label=df_test_metrics,
    overall=test_overall,
    matrices=test_matrices,
    predictions_df=df_test_predictions,
    metadata={
        "experiment": "mlp_mil_v3",
        "split": "test",
        "threshold_source": "validation",
        "thresholds": {
            bias: float(t)
            for bias, t in zip(bias_names, thresholds_v3)
        },
        "checkpoint_epoch": int(checkpoint_v3["epoch"]),
        "f_beta": 2.0,
        "biases": list(bias_names),
        "n_conversations": len(test_ids)
    }
)

for file in saved_files:
    print(file)